In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. 设置随机种子和数据长度
np.random.seed(100)
N = 500
initial_price = 100

# 2. 定义方差比 VR(q) 计算函数
def calculate_vr(prices, q=5):
    log_returns = np.diff(np.log(prices))
    n = len(log_returns)
    mu = np.mean(log_returns)
    # 1周期方差
    var_1 = np.sum((log_returns - mu)**2) / (n - 1)
    # q周期方差
    log_prices = np.log(prices)
    q_returns = log_prices[q:] - log_prices[:-q]
    m = q * (n - q + 1) * (1 - q / n)
    var_q = np.sum((q_returns - q * mu)**2) / m
    return var_q / var_1

# 3. 模拟 A 组：随机漫步 (VR 接近 1)
rw_noise = np.random.normal(0, 0.01, N)
rw_prices = initial_price * np.exp(np.cumsum(rw_noise))
vr_a = calculate_vr(rw_prices, q=5)

# 4. 模拟 B 组：正自相关趋势（VR 远大于 1）
# 收益率之间存在正相关：今天的收益率 = 0.5 * 昨天的收益率 + 噪声
ar_returns = np.zeros(N)
rho = 0.5  # 正自相关系数
for t in range(1, N):
    ar_returns[t] = rho * ar_returns[t-1] + np.random.normal(0, 0.005)
trend_prices = initial_price * np.exp(np.cumsum(ar_returns))
vr_b = calculate_vr(trend_prices, q=5)

# 5. 开始绘制视觉对比图
plt.figure(figsize=(14, 7))

# 绘制 A 组
plt.subplot(2, 1, 1)
plt.plot(rw_prices, color='#7f8c8d', label=f'Group A: Random Walk (GBM)', linewidth=1.5)
plt.title(f'Group A: VR(5) = {vr_a:.2f} (Close to 1) -> Pure Noise / Memoryless', fontsize=12, fontweight='bold')
plt.ylabel('Price')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='upper left')

# 绘制 B 组
plt.subplot(2, 1, 2)
plt.plot(trend_prices, color='#e74c3c', label=f'Group B: Momentum / Trending', linewidth=1.5)
plt.title(f'Group B: VR(5) = {vr_b:.2f} (Much Greater than 1) -> Hidden Trends / Path Persistence', fontsize=12, fontweight='bold')
plt.xlabel('Time Steps')
plt.ylabel('Price')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:

# 滚动 VR 计算
def rolling_vr(prices, q, window):
    """对价格序列逐步计算滚动 VR(q)，每次用 window 个价格点"""
    results = np.full(len(prices), np.nan)
    for i in range(window - 1, len(prices)):
        segment = prices[i - window + 1 : i + 1]
        results[i] = calculate_vr(segment, q=q)
    return results

# 三个版本的参数
configs = [
    {"q": 5,  "window": 20,  "label": "VR(5)  window=20"},
    {"q": 10, "window": 40,  "label": "VR(10) window=40"},
    {"q": 20, "window": 80,  "label": "VR(20) window=80"},
]
colors = ["#2980b9", "#27ae60", "#e67e22"]

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for cfg, color in zip(configs, colors):
    vr_a = rolling_vr(rw_prices,    q=cfg["q"], window=cfg["window"])
    vr_b = rolling_vr(trend_prices, q=cfg["q"], window=cfg["window"])
    axes[0].plot(vr_a, label=cfg["label"], color=color, linewidth=1.2)
    axes[1].plot(vr_b, label=cfg["label"], color=color, linewidth=1.2)

for ax, title in zip(axes, [
    "Group A: Random Walk — Rolling VR (VR≈1 expected)",
    "Group B: AR(1) Trending — Rolling VR (VR>1 expected)",
]):
    ax.axhline(1.0, color="black", linestyle="--", linewidth=1, alpha=0.6, label="VR=1 baseline")
    ax.set_ylabel("VR value")
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(True, linestyle="--", alpha=0.4)

axes[1].set_xlabel("Time Steps")
plt.tight_layout()
plt.show()
